# Bridge Equations for Nowcasting

This notebook demonstrates **bridge equations** — a classic approach used by central banks
worldwide (e.g., ECB, Federal Reserve, Banco Central do Brasil) to nowcast GDP using
monthly indicators.

Bridge equations "bridge" the gap between monthly indicators and the quarterly GDP by:
1. Aggregating monthly indicators to quarterly frequency
2. Estimating a simple regression of quarterly GDP on the aggregated indicators
3. Projecting missing months via AR(1) to handle partial quarters

In [ ]:
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from forecastbox.nowcasting import BridgeEquation

# Add helpers path
sys.path.insert(0, "../../utils")
from helpers import load_mixed_freq, load_macro_brazil, simulate_ragged_edge

warnings.filterwarnings("ignore")
np.random.seed(42)

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

## 1. Bridge Equation Concept

The idea is straightforward: monthly indicators like industrial production, retail sales,
and confidence surveys contain information about the current state of the economy.

A bridge equation connects these monthly indicators to the quarterly GDP target:

$$GDP_Q = \alpha + \beta_1 \bar{x}_{1,Q} + \beta_2 \bar{x}_{2,Q} + \ldots + \varepsilon_Q$$

Where $\bar{x}_{i,Q}$ is the quarterly average (or sum) of the monthly indicator $x_i$.

The key question for nowcasting: what do we do when not all months of the current
quarter are available? Bridge equations handle this by **projecting** missing months
forward (e.g., using an AR(1) model on the monthly series).

In [ ]:
# Load mixed-frequency dataset
data = load_mixed_freq()
print(f"Dataset: {data.shape[0]} monthly observations, {data.shape[1]} variables")
print(f"Date range: {data.index[0].strftime('%Y-%m')} to {data.index[-1].strftime('%Y-%m')}")

# Show frequency differences
print("\n--- Monthly indicators (available every month) ---")
monthly_cols = ["industrial_production", "retail_sales", "confidence_index"]
for col in monthly_cols:
    n_obs = data[col].notna().sum()
    print(f"  {col}: {n_obs} observations")

print("\n--- Quarterly target (only available every 3 months) ---")
gdp_obs = data["gdp_growth"].dropna()
print(f"  gdp_growth: {len(gdp_obs)} observations (quarters)")

# Visualize the frequency mismatch
fig, ax = plt.subplots(figsize=(14, 5))
for col in monthly_cols:
    ax.plot(data.index, data[col], linewidth=1, alpha=0.7, label=col.replace("_", " ").title())
ax.scatter(gdp_obs.index, gdp_obs.values, color="black", s=60, zorder=5, label="GDP Growth (Q)")
ax.set_title("Monthly Indicators vs Quarterly GDP", fontsize=14, fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Temporal Aggregation

The first step in building a bridge equation is to aggregate the monthly indicators to
quarterly frequency. Common aggregation methods:

- **Mean**: Simple average of the 3 monthly values (most common for indices)
- **Sum**: Sum of the 3 months (appropriate for flow variables like production)
- **Last**: Last month of the quarter (for stock variables)

After aggregation, we can run a simple OLS regression of GDP on the quarterly indicators.

In [ ]:
# Demonstrate temporal aggregation
monthly_indicators = data[monthly_cols]

# Aggregate to quarterly using different methods
quarterly_mean = monthly_indicators.resample("QS").mean()
quarterly_sum = monthly_indicators.resample("QS").sum()
quarterly_last = monthly_indicators.resample("QS").last()

print("Original monthly data (2023):")
print(data.loc["2023", monthly_cols].to_string())

print("\nQuarterly MEAN aggregation (2023):")
print(quarterly_mean.loc["2023"].to_string())

print("\nQuarterly SUM aggregation (2023):")
print(quarterly_sum.loc["2023"].to_string())

# Visualize aggregation effect
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, monthly_cols):
    ax.plot(data.index, data[col], "b-", alpha=0.5, linewidth=0.8, label="Monthly")
    ax.step(quarterly_mean.index, quarterly_mean[col], "r-", linewidth=2, where="mid", label="Quarterly Mean")
    ax.set_title(col.replace("_", " ").title())
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
plt.suptitle("Monthly to Quarterly Aggregation", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# Simple OLS of GDP on quarterly-aggregated indicators
gdp_quarterly = data["gdp_growth"].dropna()
common_idx = gdp_quarterly.index.intersection(quarterly_mean.index)
y = gdp_quarterly.loc[common_idx].values
X = quarterly_mean.loc[common_idx].values

# OLS with intercept
X_c = np.column_stack([np.ones(len(y)), X])
beta = np.linalg.lstsq(X_c, y, rcond=None)[0]
y_hat = X_c @ beta
r2 = 1 - np.sum((y - y_hat) ** 2) / np.sum((y - np.mean(y)) ** 2)

print(f"\nManual OLS R-squared: {r2:.4f}")
print(f"Coefficients: intercept={beta[0]:.4f}, IP={beta[1]:.4f}, RS={beta[2]:.4f}, CI={beta[3]:.4f}")

## 3. Single-Indicator Bridge

We start with the simplest case: a bridge equation using only **industrial production**
as the monthly indicator. This is a common baseline in central bank practice.

In [ ]:
# Single-indicator bridge equation: Industrial Production -> GDP
bridge_ip = BridgeEquation(
    target="gdp_growth",
    indicators=["industrial_production"],
    aggregation="mean",
    fill_method="ar1",
)

bridge_ip.fit(data)

# Show model summary
print(bridge_ip.summary())

# Nowcast
fc_ip = bridge_ip.nowcast()
print(f"\nNowcast (IP only): {fc_ip.point[0]:.4f}")
print(f"95% CI: [{fc_ip.lower_95[0]:.4f}, {fc_ip.upper_95[0]:.4f}]")
print(f"R-squared: {bridge_ip.r_squared():.4f}")

# Coefficients
print("\nCoefficients:")
print(bridge_ip.coefficients().to_string(index=False))

## 4. Multi-Indicator Bridge

We can improve the nowcast by combining **multiple indicators**. Each indicator
provides a different signal about the economy, and combining them reduces noise.

We evaluate the marginal contribution of each indicator by comparing R-squared values.

In [ ]:
# Multi-indicator bridge: all three monthly indicators
bridge_all = BridgeEquation(
    target="gdp_growth",
    indicators=["industrial_production", "retail_sales", "confidence_index"],
    aggregation="mean",
    fill_method="ar1",
)

bridge_all.fit(data)
print(bridge_all.summary())

# Nowcast with all indicators
fc_all = bridge_all.nowcast()
print(f"\nNowcast (all indicators): {fc_all.point[0]:.4f}")
print(f"95% CI: [{fc_all.lower_95[0]:.4f}, {fc_all.upper_95[0]:.4f}]")

# Compare marginal contribution of each indicator
print("\n--- Marginal Contribution Analysis ---")
indicator_names = ["industrial_production", "retail_sales", "confidence_index"]
r2_values = {}

for ind in indicator_names:
    bridge_single = BridgeEquation(
        target="gdp_growth",
        indicators=[ind],
        aggregation="mean",
    )
    bridge_single.fit(data)
    r2_values[ind] = bridge_single.r_squared()
    print(f"  {ind:30s}  R\u00b2 = {r2_values[ind]:.4f}")

r2_values["all_combined"] = bridge_all.r_squared()
print(f"  {'all_combined':30s}  R\u00b2 = {r2_values['all_combined']:.4f}")

# Plot comparison
fig, ax = plt.subplots(figsize=(10, 5))
names = [n.replace("_", "\n") for n in r2_values.keys()]
values = list(r2_values.values())
colors = ["steelblue"] * len(indicator_names) + ["darkorange"]
ax.bar(names, values, color=colors, edgecolor="black", alpha=0.8)
ax.set_ylabel("R-squared")
ax.set_title("Bridge Equation R\u00b2 by Indicator Combination", fontsize=13, fontweight="bold")
ax.grid(True, alpha=0.3, axis="y")
for i, v in enumerate(values):
    ax.text(i, v + 0.005, f"{v:.3f}", ha="center", fontsize=10)
plt.tight_layout()
plt.show()

## 5. Handling Missing Data

The real power of bridge equations for nowcasting is their ability to handle
**partial quarters** — when only 1 or 2 months of the current quarter are available.

The `fill_method` parameter controls how missing months are projected:
- `ar1`: AR(1) forecast for the remaining months
- `last`: Carry forward the last observed value
- `mean`: Fill with the series mean

As more months become available, the nowcast improves because less data is projected.

In [ ]:
# Demonstrate nowcasting with different amounts of quarterly data available
print("=" * 70)
print("Nowcast with partial quarter data")
print("=" * 70)

partial_results = []

for missing_months in [3, 2, 1, 0]:
    months_available = 3 - missing_months
    label = f"{months_available}/3 months available"

    if missing_months > 0:
        partial_data = simulate_ragged_edge(data, {
            "industrial_production": missing_months,
            "retail_sales": missing_months,
            "confidence_index": missing_months,
        })
    else:
        partial_data = data.copy()

    bridge = BridgeEquation(
        target="gdp_growth",
        indicators=indicator_names,
        aggregation="mean",
        fill_method="ar1",
    )
    bridge.fit(partial_data)
    fc = bridge.nowcast()

    partial_results.append({
        "months_available": months_available,
        "nowcast": fc.point[0],
        "lower_95": fc.lower_95[0],
        "upper_95": fc.upper_95[0],
    })
    print(f"  {label}: nowcast = {fc.point[0]:.4f}  "
          f"95% CI = [{fc.lower_95[0]:.4f}, {fc.upper_95[0]:.4f}]")

# Compare fill methods
print("\n--- Fill Method Comparison (2 missing months) ---")
partial_data_2m = simulate_ragged_edge(data, {
    "industrial_production": 2,
    "retail_sales": 2,
    "confidence_index": 2,
})

for method in ["ar1", "last", "mean"]:
    bridge_m = BridgeEquation(
        target="gdp_growth",
        indicators=indicator_names,
        aggregation="mean",
        fill_method=method,
    )
    bridge_m.fit(partial_data_2m)
    fc_m = bridge_m.nowcast()
    print(f"  fill_method='{method}': nowcast = {fc_m.point[0]:.4f}")

# Visualize nowcast convergence
results_df = pd.DataFrame(partial_results)
fig, ax = plt.subplots(figsize=(10, 5))
ax.errorbar(
    results_df["months_available"],
    results_df["nowcast"],
    yerr=[
        results_df["nowcast"] - results_df["lower_95"],
        results_df["upper_95"] - results_df["nowcast"],
    ],
    fmt="o-",
    capsize=5,
    color="steelblue",
    linewidth=2,
    markersize=8,
)
ax.set_xlabel("Months Available in Current Quarter")
ax.set_ylabel("GDP Nowcast")
ax.set_title("Bridge Equation Nowcast as Quarter Progresses", fontsize=13, fontweight="bold")
ax.set_xticks([0, 1, 2, 3])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Exercise 1: Build bridge equation for Brazilian GDP

Load the `macro_brazil.csv` dataset and build a bridge equation to nowcast
Brazilian GDP growth using inflation, interest rate, unemployment, and exchange rate
as monthly indicators. Which indicator has the highest predictive power?

In [ ]:
# Exercise 1 - Solution: Bridge equation for Brazilian GDP

# Step 1: Load and prepare Brazilian macroeconomic data
brazil = load_macro_brazil()
print(f"Brazilian macro dataset: {brazil.shape}")
print(f"Columns: {list(brazil.columns)}")
print(f"Date range: {brazil.index[0]} to {brazil.index[-1]}")
print(f"\nDescriptive statistics:")
print(brazil.describe().round(4))

# Step 2: Prepare mixed-frequency data
# GDP is available monthly but we treat it as quarterly for nowcasting
brazil_mf = brazil.copy()
# Make GDP quarterly: keep only quarter-end months (Mar, Jun, Sep, Dec)
gdp_mask = ~brazil_mf.index.month.isin([3, 6, 9, 12])
brazil_mf.loc[gdp_mask, "gdp_growth"] = np.nan
print(f"\nGDP observations after quarterly conversion: {brazil_mf['gdp_growth'].notna().sum()}")
print(f"Monthly indicator observations: {brazil_mf['inflation'].notna().sum()}")

In [ ]:
# Step 3: Build bridge equations with individual indicators
brazil_indicators = ["inflation", "interest_rate", "unemployment", "exchange_rate"]

print("=" * 70)
print("Single-Indicator Bridge Equations for Brazilian GDP")
print("=" * 70)

brazil_r2 = {}
brazil_nowcasts = {}

for ind in brazil_indicators:
    bridge_br = BridgeEquation(
        target="gdp_growth",
        indicators=[ind],
        aggregation="mean",
        fill_method="ar1",
    )
    bridge_br.fit(brazil_mf)
    fc = bridge_br.nowcast()
    brazil_r2[ind] = bridge_br.r_squared()
    brazil_nowcasts[ind] = fc.point[0]
    print(f"\n  {ind}:")
    print(f"    R-squared: {bridge_br.r_squared():.4f}")
    print(f"    Nowcast:   {fc.point[0]:.4f}")
    print(f"    95% CI:    [{fc.lower_95[0]:.4f}, {fc.upper_95[0]:.4f}]")

# Step 4: Multi-indicator bridge with inflation + interest_rate
print("\n" + "=" * 70)
print("Multi-Indicator Bridge: inflation + interest_rate")
print("=" * 70)

bridge_br_multi = BridgeEquation(
    target="gdp_growth",
    indicators=["inflation", "interest_rate"],
    aggregation="mean",
    fill_method="ar1",
)
bridge_br_multi.fit(brazil_mf)
fc_multi = bridge_br_multi.nowcast()

print(bridge_br_multi.summary())
print(f"\nNowcast: {fc_multi.point[0]:.4f}")
print(f"95% CI: [{fc_multi.lower_95[0]:.4f}, {fc_multi.upper_95[0]:.4f}]")
brazil_r2["inflation+interest_rate"] = bridge_br_multi.r_squared()

# Step 5: Full multi-indicator bridge
print("\n" + "=" * 70)
print("Full Multi-Indicator Bridge: all 4 indicators")
print("=" * 70)

bridge_br_all = BridgeEquation(
    target="gdp_growth",
    indicators=brazil_indicators,
    aggregation="mean",
    fill_method="ar1",
)
bridge_br_all.fit(brazil_mf)
fc_all_br = bridge_br_all.nowcast()

print(bridge_br_all.summary())
print(f"\nNowcast: {fc_all_br.point[0]:.4f}")
print(f"95% CI: [{fc_all_br.lower_95[0]:.4f}, {fc_all_br.upper_95[0]:.4f}]")
brazil_r2["all_combined"] = bridge_br_all.r_squared()

In [ ]:
# Step 6: Evaluate nowcast RMSE with pseudo real-time exercise
gdp_dates_br = brazil_mf["gdp_growth"].dropna().index
eval_dates_br = gdp_dates_br[-8:]  # Last 8 quarters

bridge_rt_results = []
for eval_date in eval_dates_br:
    for months_before in [3, 2, 1]:
        cutoff_idx = brazil_mf.index.get_loc(eval_date) - months_before
        if cutoff_idx < 12:
            continue
        available = brazil_mf.iloc[:cutoff_idx + 1].copy()
        available.loc[eval_date:, "gdp_growth"] = np.nan

        try:
            br_rt = BridgeEquation(
                target="gdp_growth",
                indicators=["inflation", "interest_rate"],
                aggregation="mean",
                fill_method="ar1",
            )
            br_rt.fit(available)
            fc_rt = br_rt.nowcast()
            actual = brazil_mf.loc[eval_date, "gdp_growth"]
            bridge_rt_results.append({
                "quarter": eval_date,
                "months_before": months_before,
                "nowcast": fc_rt.point[0],
                "actual": actual,
                "error": fc_rt.point[0] - actual,
            })
        except Exception:
            pass

bridge_rt_df = pd.DataFrame(bridge_rt_results)
print("\nPseudo Real-Time Bridge Equation Results:")
print(bridge_rt_df.to_string(index=False))

print("\nBridge RMSE by horizon:")
for m in sorted(bridge_rt_df["months_before"].unique()):
    subset = bridge_rt_df[bridge_rt_df["months_before"] == m]
    rmse = np.sqrt(np.mean(subset["error"] ** 2))
    print(f"  {m} months before: RMSE = {rmse:.4f}")

In [ ]:
# Step 7: Visualization of results
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: R-squared comparison
ax = axes[0]
names = [n.replace("_", "\n") for n in brazil_r2.keys()]
vals = list(brazil_r2.values())
n_single = len(brazil_indicators)
colors_bar = ["steelblue"] * n_single + ["darkorange", "forestgreen"]
ax.bar(names, vals, color=colors_bar, edgecolor="black", alpha=0.8)
ax.set_ylabel("R-squared")
ax.set_title("Bridge R\u00b2: Brazilian GDP", fontsize=12, fontweight="bold")
ax.grid(True, alpha=0.3, axis="y")
for i, v in enumerate(vals):
    ax.text(i, v + 0.005, f"{v:.3f}", ha="center", fontsize=9)
ax.tick_params(axis="x", rotation=45)

# Right: RMSE by horizon
ax = axes[1]
horizons = sorted(bridge_rt_df["months_before"].unique())
rmses = [np.sqrt(np.mean(bridge_rt_df[bridge_rt_df["months_before"] == m]["error"] ** 2))
         for m in horizons]
ax.bar(horizons, rmses, color="steelblue", edgecolor="black", alpha=0.8)
ax.set_xlabel("Months Before GDP Release")
ax.set_ylabel("RMSE")
ax.set_title("Bridge Nowcast RMSE by Horizon", fontsize=12, fontweight="bold")
ax.set_xticks(horizons)
ax.grid(True, alpha=0.3, axis="y")
for i, (h, v) in enumerate(zip(horizons, rmses)):
    ax.text(h, v + 0.002, f"{v:.4f}", ha="center", fontsize=10)

plt.tight_layout()
plt.show()

# Interpretation
best_single = max(brazil_r2.items(), key=lambda x: x[1] if x[0] in brazil_indicators else -1)
print(f"\n--- Interpretation ---")
print(f"Best single indicator: {best_single[0]} (R\u00b2 = {best_single[1]:.4f})")
print(f"Combined model R\u00b2: {brazil_r2['all_combined']:.4f}")
print(f"Adding more indicators improves fit, but marginal gains diminish.")
print(f"The bridge approach is simple, transparent, and easy to interpret.")
print(f"RMSE decreases as more months of the quarter become available,")
print(f"confirming that bridge equations improve with real-time data flow.")

### Exercise 2: Compare bridge with DFM nowcast accuracy

Run a pseudo real-time exercise comparing the bridge equation and DFM approaches.
Which method performs better at different horizons (3, 2, 1 months before GDP release)?

In [ ]:
# Exercise 2 - Solution: Compare bridge vs DFM in pseudo real-time

from forecastbox.nowcasting import DFMNowcaster

# Use the mixed_freq dataset for a fair comparison
frequency_map = {
    "industrial_production": "M",
    "retail_sales": "M",
    "confidence_index": "M",
    "gdp_growth": "Q",
}

gdp_dates = data["gdp_growth"].dropna().index
eval_dates = gdp_dates[-8:]  # Last 8 quarters

comparison_results = []

for eval_date in eval_dates:
    for months_before in [3, 2, 1]:
        cutoff_idx = data.index.get_loc(eval_date) - months_before
        if cutoff_idx < 12:
            continue

        available_data = data.iloc[:cutoff_idx + 1].copy()
        available_data.loc[eval_date:, "gdp_growth"] = np.nan
        actual = data.loc[eval_date, "gdp_growth"]

        # Bridge equation
        try:
            br = BridgeEquation(
                target="gdp_growth",
                indicators=["industrial_production", "retail_sales", "confidence_index"],
                aggregation="mean",
                fill_method="ar1",
            )
            br.fit(available_data)
            fc_br = br.nowcast()
            comparison_results.append({
                "quarter": eval_date,
                "months_before": months_before,
                "model": "Bridge",
                "nowcast": fc_br.point[0],
                "actual": actual,
                "error": fc_br.point[0] - actual,
            })
        except Exception:
            pass

        # DFM
        try:
            dfm_rt = DFMNowcaster(
                n_factors=1,
                factor_lags=2,
                frequency_map=frequency_map,
                aggregation="sum",
                em_iterations=50,
            )
            dfm_rt.fit(available_data)
            fc_dfm = dfm_rt.nowcast(target="gdp_growth")
            comparison_results.append({
                "quarter": eval_date,
                "months_before": months_before,
                "model": "DFM",
                "nowcast": fc_dfm.point[0],
                "actual": actual,
                "error": fc_dfm.point[0] - actual,
            })
        except Exception:
            pass

comp_df = pd.DataFrame(comparison_results)
print("Pseudo Real-Time Comparison: Bridge vs DFM")
print(comp_df.to_string(index=False))

In [ ]:
# Step 2: Compute RMSE by model and horizon
print("\n" + "=" * 70)
print("RMSE Comparison: Bridge vs DFM by Horizon")
print("=" * 70)

rmse_comparison = []
for model_name in ["Bridge", "DFM"]:
    model_df = comp_df[comp_df["model"] == model_name]
    for m in sorted(model_df["months_before"].unique()):
        subset = model_df[model_df["months_before"] == m]
        rmse = np.sqrt(np.mean(subset["error"] ** 2))
        mae = np.mean(np.abs(subset["error"]))
        bias = np.mean(subset["error"])
        rmse_comparison.append({
            "model": model_name,
            "horizon": f"{m} months before",
            "months_before": m,
            "RMSE": rmse,
            "MAE": mae,
            "Bias": bias,
        })

rmse_df = pd.DataFrame(rmse_comparison)
print("\n" + rmse_df[["model", "horizon", "RMSE", "MAE", "Bias"]].to_string(index=False))

# Overall RMSE
print("\n--- Overall RMSE ---")
for model_name in ["Bridge", "DFM"]:
    model_df = comp_df[comp_df["model"] == model_name]
    overall_rmse = np.sqrt(np.mean(model_df["error"] ** 2))
    print(f"  {model_name}: RMSE = {overall_rmse:.4f}")

In [ ]:
# Step 3: Visualization of comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Left: RMSE by horizon grouped by model
ax = axes[0]
horizons = sorted(rmse_df["months_before"].unique())
x = np.arange(len(horizons))
width = 0.35
bridge_rmse = [rmse_df[(rmse_df["model"] == "Bridge") & (rmse_df["months_before"] == h)]["RMSE"].values[0]
               for h in horizons]
dfm_rmse = [rmse_df[(rmse_df["model"] == "DFM") & (rmse_df["months_before"] == h)]["RMSE"].values[0]
            for h in horizons]

ax.bar(x - width / 2, bridge_rmse, width, label="Bridge", color="steelblue",
       edgecolor="black", alpha=0.8)
ax.bar(x + width / 2, dfm_rmse, width, label="DFM", color="darkorange",
       edgecolor="black", alpha=0.8)
ax.set_xlabel("Months Before GDP Release")
ax.set_ylabel("RMSE")
ax.set_title("RMSE by Horizon", fontsize=12, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels([f"{h}m" for h in horizons])
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

# Middle: Nowcast vs actual scatter
ax = axes[1]
for model_name, color in [("Bridge", "steelblue"), ("DFM", "darkorange")]:
    model_df = comp_df[comp_df["model"] == model_name]
    ax.scatter(model_df["actual"], model_df["nowcast"], color=color,
              alpha=0.6, s=40, label=model_name)
lims = [comp_df[["actual", "nowcast"]].min().min() - 0.5,
        comp_df[["actual", "nowcast"]].max().max() + 0.5]
ax.plot(lims, lims, "k--", alpha=0.5, label="Perfect forecast")
ax.set_xlabel("Actual GDP Growth")
ax.set_ylabel("Nowcast")
ax.set_title("Nowcast vs Actual", fontsize=12, fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)

# Right: Error distribution
ax = axes[2]
for model_name, color in [("Bridge", "steelblue"), ("DFM", "darkorange")]:
    model_df = comp_df[comp_df["model"] == model_name]
    ax.hist(model_df["error"], bins=10, alpha=0.5, color=color, label=model_name,
            edgecolor="black")
ax.axvline(0, color="black", linewidth=1.5)
ax.set_xlabel("Nowcast Error")
ax.set_ylabel("Frequency")
ax.set_title("Error Distribution", fontsize=12, fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Interpretation
print("\n--- Interpretation ---")
print("Bridge vs DFM comparison:")
print("- Bridge equations are simpler, faster to estimate, and more interpretable.")
print("- DFM captures common dynamics across indicators via latent factors.")
print("- At longer horizons (3 months before), DFM may outperform because it")
print("  uses the Kalman filter to optimally combine sparse information.")
print("- At shorter horizons (1 month before), bridge equations become competitive")
print("  because most data is already available and the simple OLS works well.")
print("- In practice, many central banks use both approaches and compare them.")
print("- The choice depends on: number of indicators (DFM scales better),")
print("  transparency requirements (bridge is easier to explain), and")
print("  computational resources (bridge is faster).")